# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**I don't have the FlyRank research paper itself** — it isn't something I could find or fetch,
and I'd rather leave this templated than invent two findings and critique claims that don't
actually exist in it. Drop the paper's text (or its URL, if it's fetchable) into the chat and
I'll fill this in properly. Structure to follow once you do, one block per finding:

> **Finding N: [the paper's claim, in its own words or close to it]**
> - Where the label comes from: [what event/outcome the paper is calling success/failure, and
>   what data it's derived from]
> - Does the validation design carry the claim?: [what split/holdout the paper used, if stated —
>   grouped? time-aware? random? — and whether that split matches what the claim is asserting]
> - My question, constructive tone: [one specific, answerable methodology question — not "this
>   is wrong," but "how was X handled?" or "does Y column exist in the training data too?"]

The kind of question this lane's own work already raises, as a starting point once the real
findings are in hand: does the paper's headline metric survive a client-grouped split the way
`w05_model` checks for here in Section 2? Is there a column in its feature set that plays the
same role `pct_change` played in the `w03_data_contract` leakage trap?

In [ ]:
# Placeholder — nothing to run until the paper's actual findings are filled in above.
print("Section 1 needs the real FlyRank paper text before this cell can check anything against it.")

## 2. My model under an honest split (before/after)

`w05_model` already used a client-grouped split as its primary design, reasoned out in that
notebook's Section 2. The honest "before" here is the split most people reach for first — a
plain random row-level split, the same shape `w03_data_contract`'s quick demo used — rebuilt
side by side with the grouped "after" so the gap between them is a real, computed number instead
of an assertion.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"

raw = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        MODE(ga4_data_available)                                       AS ga4_data_available,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["ctr"] = (df["total_clicks"] / df["total_impressions"]).round(4)
df["impressions_per_active_day"] = (df["total_impressions"] / df["active_days"]).round(2)
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)
df["position_volatility"] = df["position_volatility"].fillna(0.0)
ga4_dummies = pd.get_dummies(df["ga4_data_available"], prefix="ga4", dummy_na=True)
df = pd.concat([df, ga4_dummies], axis=1)
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["is_declining"] = (df["pct_change"] < -0.2).astype(int)

FINAL_FEATURES = (["total_impressions", "total_clicks", "avg_position", "active_days",
                    "position_volatility", "volatility_is_filled", "ctr",
                    "impressions_per_active_day"] + list(ga4_dummies.columns))

def fit_and_score(train_df, test_df):
    scaler = StandardScaler().fit(train_df[FINAL_FEATURES])
    clf = LogisticRegression(max_iter=1000).fit(
        scaler.transform(train_df[FINAL_FEATURES]), train_df["is_declining"])
    return roc_auc_score(test_df["is_declining"],
                          clf.predict_proba(scaler.transform(test_df[FINAL_FEATURES]))[:, 1])

# BEFORE — plain random row-level split, same shape as w03_data_contract's quick demo
before_train, before_test = train_test_split(df, test_size=0.3, random_state=42, stratify=df["is_declining"])
before_auc = fit_and_score(before_train, before_test)

# AFTER — client-grouped split, same design as w05_model
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
after_train, after_test = df.iloc[train_idx], df.iloc[test_idx]
after_auc = fit_and_score(after_train, after_test)

print(f"BEFORE — random split AUC:          {before_auc:.3f}")
print(f"AFTER  — client-grouped split AUC:  {after_auc:.3f}")
print(f"Gap:                                 {before_auc - after_auc:+.3f}")
print("\nA gap here (before > after) means the random split was letting the model partly learn")
print("client identity rather than a signal that generalizes to a client it hasn't seen.")

## 3. Leakage audit

Same hunt as `w03_feature_leakage_check`, re-run on the final feature set actually used to
train — `FINAL_FEATURES` above, evaluated on the grouped test split from Section 2 so the
numbers reflect the same honest split the model itself was scored on.

In [ ]:
checked = after_test.copy()

# Attack 1 — single-feature AUC scan, nothing should sit near 1.0
single_feature_auc = {}
for col in FINAL_FEATURES:
    d = checked.dropna(subset=[col, "is_declining"])
    if d[col].nunique() < 2:
        continue
    auc = roc_auc_score(d["is_declining"], d[col])
    single_feature_auc[col] = round(max(auc, 1 - auc), 3)

print("Single-feature AUC against is_declining, grouped test split:")
print(pd.Series(single_feature_auc).sort_values(ascending=False))

# Attack 2 — the same trap, re-run on the final feature set and the honest grouped split
checked["second_half_share"] = checked["imp_second_half"] / (checked["imp_first_half"] + checked["imp_second_half"])

def quick_auc(cols, train_data, test_data):
    scaler = StandardScaler().fit(train_data[cols])
    clf = LogisticRegression(max_iter=1000).fit(scaler.transform(train_data[cols]), train_data["is_declining"])
    return roc_auc_score(test_data["is_declining"], clf.predict_proba(scaler.transform(test_data[cols]))[:, 1])

after_train_trap = after_train.copy()
after_train_trap["second_half_share"] = (
    after_train_trap["imp_second_half"] / (after_train_trap["imp_first_half"] + after_train_trap["imp_second_half"]))

print(f"\nHonest AUC (final feature set, grouped split): {after_auc:.3f}")
leaked_auc = quick_auc(FINAL_FEATURES + ["second_half_share"], after_train_trap, checked)
print(f"Leaked AUC (adding second_half_share back in):  {leaked_auc:.3f}")
print("\nsecond_half_share deleted — it only ever existed to confirm the leak is still catchable.")
del checked["second_half_share"]

## 4. Claim rewrite

**Boldest draft sentence, the kind that's easy to write after a table like Section 2's:**
> "This model predicts which pages will decline, and the grouped split proves it generalizes to
> new clients."

**Problems with it:** "predicts... will decline" claims a forecast this single-month,
within-month label was never validated to make — there's no next-month data behind that verb.
"Proves it generalizes" is doing more work than one grouped split on one month can support; it's
one piece of evidence, not proof, and it says nothing about generalizing across time, only
across clients.

**Rewrite, in the paper's required register:**
> "On this month's data, the model's score is directionally associated with within-month
> decline as observed in a client-grouped holdout — a stronger check than a random split, but
> still a single month and a single split. Treat the score as decision-support for prioritizing
> review, not as a forecast of future performance."`

In [ ]:
claims = pd.DataFrame([
    {"version": "draft (bold)",
     "text": "This model predicts which pages will decline, and the grouped split proves it generalizes to new clients."},
    {"version": "rewrite (safe)",
     "text": ("On this month's data, the model's score is directionally associated with within-month decline as "
              "observed in a client-grouped holdout — a stronger check than a random split, but still a single "
              "month and a single split. Treat the score as decision-support for prioritizing review, not as a "
              "forecast of future performance.")},
])
pd.set_option("display.max_colwidth", None)
claims

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.